# jpc FC-ResNet Comparison Demo

Replicates the FC-ResNet architecture from the jpc library (thebuckleylab/jpc)
using FabricPC's predictive coding mechanics, enabling a fair comparison of the
two implementations.

The PreActResBlock bundles a linear path and an identity skip connection inside one node:

```
z_mu = scale * matmul(act_x, W) + x   # linear + unscaled skip
```

In FabricPC's graph-based architecture, this summation would be an explicit IdentityNode with in_degree=2,
and mupc.py:229 would compute `a = gain / sqrt(fan_in * K)` with K=2 for each incoming edge.
Effective scales folded into the monolithic PreActResBlock:
- Linear path: `a_linear * a_identity = sqrt(2)/sqrt(N) * 1/sqrt(2) = 1/sqrt(N)`
- Skip path: `a_identity = 1/sqrt(2)`

## Architecture

Matching jpc mupc.ipynb:

```
input(784) -> FCInput(width) -> PreActResBlock(width) x (depth-2) -> Readout(10)
```

Each PreActResBlock computes:
```
z_mu = hidden_scale * (W @ act(x)) + x     (pre-activation + identity skip)
```

## Scaling Modes (`--scaling` flag)

| Mode | in | hidden | out | skip |
|------|----|--------|-----|------|
| `jpc` | 1/√D | 1/√(N·L) | 1/N | 1 |
| `fabricpc` | 1/√D | gain/√(N·K) | 1/N | 1/√K (K=2 in_degree — BROKEN at depth>16) |
| `fabricpc_v2` | 1/√D | gain/√(N·L) | 1/N | 1 (depth L + Kaiming gain) |

Key difference from jpc: FabricPC uses fixed-step SGD inference with gradient
norm clipping (`InferenceSGDNormClip`), while jpc uses an adaptive ODE solver.

## Results (3 epochs, MNIST, jpc scaling, width=128)

- depth=5: ~93% (matches jpc reference)
- depth=10: ~93%
- depth=30: ~86% (limited by SGD inference convergence)

| Depth | jpc scaling | fabricpc scaling | fabricpc_v2 scaling |
|-------|-------------|------------------|---------------------|
| 8     | ~90.8%      | ~92.6%           | ~91.6%              |
| 16    | ~87.8%      | ~85.6%           | ~88.4%              |
| 32    | ~85.2%      | ~43.4%           | ~86.8%              |
| 64    | ~83.5%      | ~11.5%           | ~85.4%              |
| 128   | ~83.9%      | ~10.3%           | (not tested)        |

Root cause of fabricpc collapse: `skip_scale = 1/sqrt(2)` causes
exponential signal decay through the identity path. Over L layers, the
coherent signal decays as 0.707^L. Fix (fabricpc_v2): `skip_scale = 1.0`.

## Imports & Setup

In [ ]:
import math
import time
from typing import Tuple

import jax
import jax.numpy as jnp
import numpy as np
import optax

from fabricpc.nodes import IdentityNode
from fabricpc.nodes.base import NodeBase, SlotSpec
from fabricpc.core.topology import Edge
from fabricpc.graph_assembly import TaskMap, graph
from fabricpc.graph_initialization import initialize_params
from fabricpc.core.activations import IdentityActivation, ReLUActivation
from fabricpc.core.energy import GaussianEnergy
from fabricpc.core.inference import InferenceSGDNormClip
from fabricpc.core.initializers import NormalInitializer, initialize
from fabricpc.core.types import NodeParams
from fabricpc.training import train_pcn, evaluate_pcn
from fabricpc.utils.data.dataloader import MnistLoader
from fabricpc import setup_jax

setup_jax()
jax.config.update("jax_default_prng_impl", "threefry2x32")

## Scaling Factor Computation

In [ ]:
def compute_scaling_factors(input_dim, width, depth, mode):
    """Compute per-layer-type scaling factors."""
    if mode == "jpc":
        in_scale = 1.0 / math.sqrt(input_dim)
        hidden_scale = 1.0 / math.sqrt(width * depth)
        skip_scale = 1.0
        out_scale = 1.0 / width
    elif mode == "fabricpc":
        K = 2
        relu_gain = math.sqrt(2.0)
        in_scale = 1.0 / math.sqrt(input_dim)
        hidden_scale = relu_gain / math.sqrt(width * K)
        skip_scale = 1.0 / math.sqrt(K)
        out_scale = 1.0 / width
    elif mode == "fabricpc_v2":
        relu_gain = math.sqrt(2.0)
        in_scale = 1.0 / math.sqrt(input_dim)
        hidden_scale = relu_gain / math.sqrt(width * depth)
        skip_scale = 1.0
        out_scale = 1.0 / width
    else:
        raise ValueError(f"Unknown scaling mode: {mode!r}.")
    return in_scale, hidden_scale, out_scale, skip_scale

## Custom Nodes

Pre-activation FC-ResNet blocks mirroring the jpc library's implementation.

In [ ]:
class FCInputNode(NodeBase):
    """First layer of the FC-ResNet: z_mu = in_scale * (W @ x). No activation, no skip."""

    def __init__(self, shape, name, activation=IdentityActivation(), energy=GaussianEnergy(), latent_init=NormalInitializer(), weight_init=NormalInitializer(mean=0.0, std=1.0), scale=1.0):
        super().__init__(shape=shape, name=name, activation=activation, energy=energy, latent_init=latent_init, weight_init=weight_init, scale=scale, use_bias=False)

    @staticmethod
    def get_slots():
        return {"in": SlotSpec(name="in", is_multi_input=False)}

    @staticmethod
    def get_variance_factor(source_shape, config, weight_init):
        return float(np.prod(source_shape))

    @staticmethod
    def initialize_params(key, node_shape, input_shapes, weight_init=None, config=None):
        if config is None:
            config = {}
        if weight_init is None:
            weight_init = NormalInitializer(mean=0.0, std=1.0)
        weights_dict = {}
        keys = jax.random.split(key, len(input_shapes))
        for i, (edge_key, in_shape) in enumerate(input_shapes.items()):
            in_features = int(np.prod(in_shape))
            out_features = int(np.prod(node_shape))
            weight_shape = (in_features, out_features)
            weights_dict[edge_key] = initialize(keys[i], weight_shape, weight_init)
        return NodeParams(weights=weights_dict, biases={})

    @staticmethod
    def forward(params, inputs, state, node_info):
        config = node_info.node_config
        scale = config.get("scale", 1.0)
        batch_size = state.z_latent.shape[0]
        edge_key, x = next(iter(inputs.items()))
        x_flat = x.reshape(batch_size, -1)
        W = params.weights[edge_key]
        z_mu = scale * jnp.matmul(x_flat, W)
        error = state.z_latent - z_mu
        state = state._replace(z_mu=z_mu, error=error)
        node_class = node_info.node_class
        state = node_class.energy_functional(state, node_info)
        return state


class PreActResBlock(NodeBase):
    """Pre-activation residual block: z_mu = hidden_scale * (W @ act(x)) + skip_scale * x"""

    def __init__(self, shape, name, activation=ReLUActivation(), energy=GaussianEnergy(), latent_init=NormalInitializer(), weight_init=NormalInitializer(mean=0.0, std=1.0), scale=1.0, skip_scale=1.0):
        super().__init__(shape=shape, name=name, activation=activation, energy=energy, latent_init=latent_init, weight_init=weight_init, scale=scale, skip_scale=skip_scale, use_bias=False)

    @staticmethod
    def get_slots():
        return {"in": SlotSpec(name="in", is_multi_input=False)}

    @staticmethod
    def get_variance_factor(source_shape, config, weight_init):
        return float(source_shape[-1])

    @staticmethod
    def initialize_params(key, node_shape, input_shapes, weight_init=None, config=None):
        if config is None:
            config = {}
        if weight_init is None:
            weight_init = NormalInitializer(mean=0.0, std=1.0)
        weights_dict = {}
        keys = jax.random.split(key, len(input_shapes))
        for i, (edge_key, in_shape) in enumerate(input_shapes.items()):
            in_features = in_shape[-1]
            out_features = node_shape[-1]
            weight_shape = (in_features, out_features)
            weights_dict[edge_key] = initialize(keys[i], weight_shape, weight_init)
        return NodeParams(weights=weights_dict, biases={})

    @staticmethod
    def forward(params, inputs, state, node_info):
        config = node_info.node_config
        scale = config.get("scale", 1.0)
        skip_scale = config.get("skip_scale", 1.0)
        activation = node_info.activation
        edge_key, x = next(iter(inputs.items()))
        W = params.weights[edge_key]
        act_x = type(activation).forward(x, activation.config)
        z_mu = scale * jnp.matmul(act_x, W) + skip_scale * x
        error = state.z_latent - z_mu
        state = state._replace(z_mu=z_mu, error=error)
        node_class = node_info.node_class
        state = node_class.energy_functional(state, node_info)
        return state


class PreActReadout(NodeBase):
    """Pre-activation output layer: z_mu = out_scale * (W @ act(x)). No skip."""

    def __init__(self, shape, name, activation=ReLUActivation(), energy=GaussianEnergy(), latent_init=NormalInitializer(), weight_init=NormalInitializer(mean=0.0, std=1.0), scale=1.0):
        super().__init__(shape=shape, name=name, activation=activation, energy=energy, latent_init=latent_init, weight_init=weight_init, scale=scale, use_bias=False)

    @staticmethod
    def get_slots():
        return {"in": SlotSpec(name="in", is_multi_input=False)}

    @staticmethod
    def get_variance_factor(source_shape, config, weight_init):
        return float(source_shape[-1])

    @staticmethod
    def initialize_params(key, node_shape, input_shapes, weight_init=None, config=None):
        if config is None:
            config = {}
        if weight_init is None:
            weight_init = NormalInitializer(mean=0.0, std=1.0)
        weights_dict = {}
        keys = jax.random.split(key, len(input_shapes))
        for i, (edge_key, in_shape) in enumerate(input_shapes.items()):
            in_features = in_shape[-1]
            out_features = int(np.prod(node_shape))
            weight_shape = (in_features, out_features)
            weights_dict[edge_key] = initialize(keys[i], weight_shape, weight_init)
        return NodeParams(weights=weights_dict, biases={})

    @staticmethod
    def forward(params, inputs, state, node_info):
        config = node_info.node_config
        scale = config.get("scale", 1.0)
        activation = node_info.activation
        edge_key, x = next(iter(inputs.items()))
        W = params.weights[edge_key]
        act_x = type(activation).forward(x, activation.config)
        z_mu = scale * jnp.matmul(act_x, W)
        error = state.z_latent - z_mu
        state = state._replace(z_mu=z_mu, error=error)
        node_class = node_info.node_class
        state = node_class.energy_functional(state, node_info)
        return state

## FC-ResNet Graph Builder

In [ ]:
def build_fc_resnet(width, depth, scaling_mode, eta_infer, infer_steps, max_norm, input_dim=784, output_dim=10):
    """Build a fully-connected ResNet matching jpc's mupc.ipynb architecture."""
    if infer_steps is None:
        infer_steps = 3 * depth

    in_scale, hidden_scale, out_scale, skip_scale = compute_scaling_factors(
        input_dim, width, depth, scaling_mode
    )

    weight_init = NormalInitializer(mean=0.0, std=1.0)

    input_node = IdentityNode(shape=(input_dim,), name="input")

    layer_0 = FCInputNode(
        shape=(width,), name="layer_0", weight_init=weight_init, scale=in_scale,
    )

    all_nodes = [input_node, layer_0]
    all_edges = [Edge(source=input_node, target=layer_0.slot("in"))]

    prev = layer_0
    for i in range(1, depth - 1):
        block = PreActResBlock(
            shape=(width,), name=f"layer_{i}", weight_init=weight_init,
            scale=hidden_scale, skip_scale=skip_scale,
        )
        all_nodes.append(block)
        all_edges.append(Edge(source=prev, target=block.slot("in")))
        prev = block

    readout = PreActReadout(
        shape=(output_dim,), name="output", weight_init=weight_init, scale=out_scale,
    )
    all_nodes.append(readout)
    all_edges.append(Edge(source=prev, target=readout.slot("in")))

    structure = graph(
        nodes=all_nodes,
        edges=all_edges,
        task_map=TaskMap(x=input_node, y=readout),
        inference=InferenceSGDNormClip(eta_infer=eta_infer, infer_steps=infer_steps, max_norm=max_norm),
    )

    return structure

## Variance Diagnostics (Optional)

In [ ]:
def probe_variance(params, structure, test_loader, rng_key):
    """Measure per-layer variance diagnostics after inference converges."""
    from fabricpc.graph_initialization.state_initializer import initialize_graph_state
    from fabricpc.core.inference import run_inference

    batch = next(iter(test_loader))
    x_batch, y_batch = batch
    x_batch = jnp.array(x_batch)
    y_batch = jnp.array(y_batch)
    batch_size = x_batch.shape[0]

    task_map = structure.task_map
    clamps = {task_map["x"]: x_batch, task_map["y"]: y_batch}

    state = initialize_graph_state(structure, batch_size, rng_key, clamps=clamps, params=params)
    final_state = run_inference(params, state, clamps, structure)

    node_order = list(structure.nodes.keys())
    print("\n" + "=" * 72)
    print("Per-Layer Variance Diagnostics (after inference)")
    print("=" * 72)
    print(f"{'Layer':<12} {'Var(z_mu)':>12} {'Var(z_lat)':>12} {'Var(grad)':>12} {'Var(error)':>12}")
    print("-" * 72)

    for node_name in node_order:
        ns = final_state.nodes[node_name]
        z_mu_var = float(jnp.var(ns.z_mu))
        z_lat_var = float(jnp.var(ns.z_latent))
        grad_var = float(jnp.var(ns.latent_grad))
        error_var = float(jnp.var(ns.error))
        print(f"{node_name:<12} {z_mu_var:>12.4f} {z_lat_var:>12.4f} {grad_var:>12.4f} {error_var:>12.4f}")
    print("=" * 72)

## Configuration

In [ ]:
scaling = "jpc"      # "jpc", "fabricpc", or "fabricpc_v2"
depth = 10           # Total parameterized layers
width = 128          # Hidden layer width
batch_size = 256     # Batch size
num_epochs = 3       # Training epochs
eta_infer = 0.2      # Inference rate
param_lr = 0.001     # Parameter learning rate
infer_steps = None   # None = 3*depth
max_norm = 0.2       # Gradient norm clipping
verbose = False
probe_variance_flag = False  # Set True to run variance diagnostics after training

## Build Model

In [ ]:
print("=" * 60)
print("jpc FC-ResNet Comparison on MNIST")
print("=" * 60)
print(f"Scaling mode: {scaling}")
print(f"Architecture: FC-ResNet, depth={depth}, width={width}")
steps = infer_steps or 3 * depth
print(f"Inference: eta={eta_infer}, steps={steps}, max_norm={max_norm}")
print(f"Training: lr={param_lr}, batch_size={batch_size}, epochs={num_epochs}")

master_rng_key = jax.random.PRNGKey(42)
graph_key, train_key, eval_key = jax.random.split(master_rng_key, 3)

structure = build_fc_resnet(
    width=width,
    depth=depth,
    scaling_mode=scaling,
    eta_infer=eta_infer,
    infer_steps=infer_steps,
    max_norm=max_norm,
)

params = initialize_params(structure, graph_key)
total_params = sum(p.size for p in jax.tree_util.tree_leaves(params))
print(f"Model: {len(structure.nodes)} nodes, total_params={total_params:,}")

train_loader = MnistLoader("train", batch_size=batch_size, tensor_format="flat", shuffle=True, seed=42)
test_loader = MnistLoader("test", batch_size=batch_size, tensor_format="flat", shuffle=False)

optimizer = optax.adamw(param_lr, weight_decay=0.01)
train_config = {"num_epochs": num_epochs}

## Train

In [ ]:
print(f"\nTraining for {num_epochs} epochs (JIT on first batch)...")
start_time = time.time()

trained_params, energy_history, _ = train_pcn(
    params=params,
    structure=structure,
    train_loader=train_loader,
    optimizer=optimizer,
    config=train_config,
    rng_key=train_key,
    verbose=verbose,
)

elapsed = time.time() - start_time
print(f"Training time: {elapsed:.1f}s ({elapsed / num_epochs:.1f}s per epoch)")

## Evaluate

In [ ]:
print("\nEvaluating...")
metrics = evaluate_pcn(trained_params, structure, test_loader, train_config, eval_key)
accuracy = metrics["accuracy"] * 100
print(f"Test Accuracy: {accuracy:.2f}%")

# Optional: variance diagnostics
if probe_variance_flag:
    probe_variance(trained_params, structure, test_loader, eval_key)